# 01 · 强化学习对齐总览与 RLHF 全景

> 前置:建议先看过 `test-lora` 的 `04_衍生_从SFT走向强化学习`,或至少知道 SFT(监督微调)是什么。
> 本 notebook **只讲概念、建立心智地图**,几乎不跑代码(只有一个几十行的玩具「策略梯度」小例子)。

回顾一下:`test-lora` 里我们做的都是 **SFT** —— 给模型看「问题 → 标准答案」,让它模仿。
但 SFT 只会模仿,学不到「答案 A 比答案 B 好」这种**偏好**,也没针对「自己生成的、可能有瑕疵的前缀」优化过。
**强化学习(RL)** 就是用来突破这个天花板的框架。这一章我们把 RL 对齐的每个零件讲清楚。

## 一、把强化学习的词汇「翻译」到语言模型上

强化学习最初是用来玩游戏、控制机器人的,它的核心词汇是:**状态 (state)**、**动作 (action)**、**奖励 (reward)**、**策略 (policy)**。
在大模型对齐里,这些词有一个非常自然的对应:

| 强化学习术语 | 在语言模型里对应什么 |
| --- | --- |
| 状态 state \(s_t\) | 到目前为止的上下文:prompt + 已经生成的那部分文字 |
| 动作 action \(a_t\) | 下一个要生成的 token |
| 策略 policy \(\pi_\theta\) | 语言模型本身(给定上下文,输出下一个 token 的概率分布) |
| 轨迹 trajectory | 一整段完整的回答(从 prompt 一路采样到结束) |
| 奖励 reward \(r\) | 对**整段回答**好坏的打分(通常只在结尾给一次,很「稀疏」) |

**一句话:** 生成一段回答 = 一条轨迹;我们希望调整策略 \(\pi_\theta\)(也就是模型权重),
让「能拿到高奖励的回答」出现的概率变大。这就是策略优化。

## 二、强化学习到底在优化什么

目标很直白:**最大化期望奖励**。写成公式:

\[ \max_{\theta}\; \mathbb{E}_{x\sim D,\; y\sim \pi_\theta(\cdot\mid x)}\big[\, r(x, y) \,\big] \]

其中 \(x\) 是 prompt,\(y\) 是模型按当前策略采样出来的回答,\(r(x,y)\) 是这段回答的奖励。

和 SFT 的关键区别:

- **SFT**:有「标准答案」,直接用交叉熵让模型去逼近它(监督学习)。
- **RL**:没有标准答案,只有一个「打分器」。模型自己先**生成**,再根据**分数**去调整 —— 这叫「试错 (trial and error)」。

这就带来一个问题:奖励 \(r\) 通常不可导(它可能来自人类打分、另一个模型、或一段规则代码),
没法像 SFT 那样直接对它求梯度。**策略梯度 (Policy Gradient)** 就是解决这个问题的钥匙。

## 三、策略梯度的直觉:让好动作更可能发生

策略梯度定理给出一个可以直接用采样来估计的梯度:

\[ \nabla_\theta\, \mathbb{E}[r] \;=\; \mathbb{E}\big[\, r \cdot \nabla_\theta \log \pi_\theta(y\mid x) \,\big] \]

直觉上非常好理解:

- 如果一段回答 \(y\) 拿到了**高奖励**,就沿着「让这段回答更可能出现」的方向推一把(提高 \(\log \pi_\theta(y\mid x)\));
- 如果奖励**低**(甚至为负),就往反方向推,让它更不可能出现。

最朴素的实现就是 **REINFORCE**:`loss = - r * log π(y|x)`。
下面用一个 3 个动作的玩具例子,亲眼看看「高奖励动作的概率如何被慢慢推高」。

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

# 三个动作的"真实好坏"(奖励):动作 2 最好,动作 0 最差
true_reward = torch.tensor([0.0, 1.0, 3.0])

# 策略参数:每个动作一个 logit,初始相等 => 均匀策略(各 1/3)
logits = torch.zeros(3, requires_grad=True)
opt = torch.optim.SGD([logits], lr=0.1)

for step in range(60):
    probs = F.softmax(logits, dim=-1)
    a = torch.multinomial(probs, 1).item()   # 按当前策略采样一个动作
    r = true_reward[a]                        # 环境给出奖励
    loss = -(r * torch.log(probs[a]))         # REINFORCE:让高奖励动作概率上升
    opt.zero_grad(); loss.backward(); opt.step()
    if (step + 1) % 15 == 0:
        p = F.softmax(logits, dim=-1).detach()
        print(f"step {step+1:>2}: 策略概率 = [{p[0]:.2f}, {p[1]:.2f}, {p[2]:.2f}]  (动作2应最高)")

print("\n可以看到:纯靠"采样 + 按奖励调整概率",策略逐渐学会偏向最优动作。")
print("这就是所有策略梯度类方法(含 PPO / GRPO)的最小内核。")

## 四、朴素策略梯度的两个麻烦,和 PPO 的两把「安全锁」

朴素 REINFORCE 能用,但训练大模型时有两个致命问题:

1. **方差太大**:奖励是稀疏的、采样是随机的,梯度估计噪声很大 → 训练很抖。
   - 解法:引入 **baseline / 优势 (advantage)**。用「这次的奖励 - 平均水平」代替原始奖励,只关心「比平均好还是差」。这需要一个 **价值网络 (value / critic)** 来估计「平均水平」。
2. **步子迈太大会崩**:一次更新如果把策略推得离原来太远,模型可能直接说胡话,再也回不来。
   - 解法:**PPO 的裁剪 (clip)**,限制「新策略 / 旧策略」的比值不许偏离 1 太多;
   - 以及 **KL 惩罚**:惩罚策略偏离一个「参考模型」太远。

PPO 的目标函数(简化版)长这样:

\[ \mathcal{L}^{PPO} = \mathbb{E}\Big[\min\big(\rho_t A_t,\; \mathrm{clip}(\rho_t, 1-\epsilon, 1+\epsilon)\,A_t\big)\Big] - \beta\, \mathrm{KL}(\pi_\theta \Vert \pi_{ref}) \]

其中 \(\rho_t = \frac{\pi_\theta(a_t)}{\pi_{old}(a_t)}\) 是概率比,\(A_t\) 是优势,\(\epsilon\) 是裁剪范围。
你现在不需要背公式,只要记住两把安全锁:**clip(别一步迈太大)** 和 **KL 惩罚(别跑太偏)**。

## 五、RLHF 全景:主流大模型是怎么对齐的

RLHF = Reinforcement Learning from **Human Feedback**(从人类反馈中做强化学习)。
它的经典流程分三步:**SFT → 训练奖励模型 → 用 RL 优化策略**。

```mermaid
flowchart TD
    pretrain["预训练底座<br/>(会说话,不会聊天)"] --> sft["第一步:SFT<br/>test-lora 做的<br/>LoRA/QLoRA 指令微调"]
    sft --> collect["收集人类偏好<br/>对同一问题的多个回答排序"]
    collect --> rm["第二步:训练奖励模型 RM<br/>学会给回答打分<br/>(本仓库 02)"]
    sft --> policy["策略模型 Policy<br/>初始 = SFT 模型"]
    rm --> rl["第三步:强化学习<br/>PPO(04)/ GRPO(05)"]
    policy --> rl
    rl --> aligned["对齐后的模型<br/>更符合人类偏好"]
    sft -. 也可跳过 RM 直接学 .-> dpo["DPO(本仓库 03)<br/>直接从成对偏好优化"]
    collect -. 成对偏好数据 .-> dpo
    dpo --> aligned
```

四个关键角色,务必分清:

- **策略模型 (Policy)**:我们要优化的语言模型,初始就是 SFT 得到的模型。
- **参考模型 (Reference)**:通常是**冻结**的 SFT 模型,用来算 KL 惩罚、约束策略「别跑太偏」。
- **奖励模型 (Reward Model)**:输入「问题+回答」,输出一个分数,代表人类有多喜欢。
- **价值模型 (Value / Critic)**:PPO 专用,估计「当前状态往后大概能拿多少奖励」,用来降方差。

## 六、三条主流路线对比

| 路线 | 要不要单独的奖励模型 | 要不要在线采样 | 同时维护几个模型 | 稳定性/资源 | 适合场景 | 本仓库 |
| --- | --- | --- | --- | --- | --- | --- |
| **RM + PPO** | 要(先训 RM) | 要(边生成边学) | 4:策略/参考/奖励/价值 | 复杂、吃显存、易抖 | 经典 RLHF,上限高 | 04 |
| **DPO** | **不要** | **不要**(离线) | 2:策略/参考 | 稳、省资源 | 有成对偏好数据,**入门首选** | 03 |
| **GRPO** | 用**奖励函数**替代 | 要 | 2~3:策略/参考[/奖励] | 中等 | 有**可验证**奖励(数学/代码/格式) | 05 |

一句话记住每条路线:

- **PPO**:「在线试错 + 打分器打分 + 两把安全锁」,最经典也最重。
- **DPO**:数学上证明了「可以不训奖励模型、不做采样」,直接用一条 `(问题, 好回答 chosen, 坏回答 rejected)` 的损失把 chosen 相对 rejected 的概率拉高。**又稳又省,和 LoRA 天生一对。**
- **GRPO**:PPO 的「瘦身版」。去掉价值网络,改用「同一个问题采样一组回答,用组内的相对好坏当优势」。特别适合答案能被自动验证的任务(比如数学题对不对)。

## 七、本仓库的路线图与显存预期

我们会按「由简到繁、由稳到难」的顺序走:

1. **02 奖励模型**:先造一个「打分器」,它是 PPO 的燃料,也是最后评估的裁判。
2. **03 DPO**(推荐先掌握):最稳、最省,直接在 4-bit 底座 + LoRA 上做偏好对齐。
3. **04 PPO**:走一遍经典 RLHF 闭环,亲手感受 reward 上升、KL 被约束。
4. **05 GRPO**:面向可验证任务(算术题),体会「组内相对优势 + 自定义奖励函数」。
5. **06 评估**:用 02 的奖励模型当裁判,对比对齐前后的胜率、平均分与 KL。

| 章节 | 方法 | 底座 | 精度 | 大致显存 |
| --- | --- | --- | --- | --- |
| 02 | Reward Model + LoRA | Qwen2.5-0.5B | bf16 | ~3-6GB |
| 03 | DPO + QLoRA | Qwen2.5-1.5B | 4-bit | ~8-12GB |
| 04 | PPO | Qwen2.5-0.5B | bf16 | ~8-14GB |
| 05 | GRPO | Qwen2.5-0.5B | bf16 | ~8-14GB |

> 所有 notebook 都把步数/批量/序列长度设得很小,**保证单张 16GB 能真跑通**;想要明显效果就照注释调大。

## 小结

- 强化学习对齐的本质:**没有标准答案,只有打分器;模型先生成、再按分数调整策略**(策略梯度)。
- 四个角色:策略 / 参考 / 奖励 / 价值;两把安全锁:**clip** 与 **KL 惩罚**。
- 三条路线:**PPO**(经典重)、**DPO**(稳省,入门首选)、**GRPO**(面向可验证任务)。
- 承上启下:`test-lora` 的 SFT 模型就是这里策略模型的**起点**。

下一站:[`02_奖励模型RewardModel原理与实战`](02_奖励模型RewardModel原理与实战.ipynb),动手训练第一个「打分器」。